In [ ]:
import numpy as np
import matplotlib.pyplot as plt

G = 1.0
M_star = 1.0
dt = 0.01
N = 8000


R_inner = 6
R_outer = 50

m_particle = 1e-4


# VISCOSITY
nu_visc0 = 0.0002

# PRESSURE 
cs = 0.01   # sound speed


# SECOND BODY
M2 = 0.01
R2i = 15.0

r2 = R2i
Omega2 = np.sqrt(G * M_star / r2**3)

def bh2_position(t):
    x = r2 * np.cos(Omega2 * t)
    y = r2 * np.sin(Omega2 * t)
    vx = -r2 * Omega2 * np.sin(Omega2 * t)
    vy =  r2 * Omega2 * np.cos(Omega2 * t)
    return x, y, vx, vy



# DISK INITIALIZATION
def initialize_disk(m_particle=1e-4, N=5000):
    r = np.random.uniform(R_inner, R_outer, N)
    theta = np.random.uniform(0, 2*np.pi, N)

    x = r * np.cos(theta)
    y = r * np.sin(theta)

    v = np.sqrt(G * M_star / r)
    vx = -v * np.sin(theta)
    vy =  v * np.cos(theta)

    m = np.ones(N) * m_particle
    return x, y, vx, vy, m



# DENSITY PROXY

def local_density_proxy(x, y):
    r = np.sqrt(x**2 + y**2)
    rho = 1.0 / (r + 1e-2)
    return rho


# PRESSURE ACCELERATION 

def pressure_acceleration(x, y):
    r = np.sqrt(x**2 + y**2)

    # log-density gradient in radial approximation
    # rho ~ 1/r → ln rho ~ -ln r
    # grad ln rho ~ -1/r * r_hat
    ax = cs**2 * x / (r**2 + 1e-8)
    ay = cs**2 * y / (r**2 + 1e-8)

    return ax, ay



# BONDI RADIUS

def bondi_radius(M, v_rel):
    return G * M / (v_rel**2 + cs**2)



# MAIN STEP
def step(x, y, vx, vy, m, t):
    global M2

    x2, y2, vx2, vy2 = bh2_position(t)


    # star gravity
    r = np.sqrt(x**2 + y**2)
    ax = -G * M_star * x / (r**3 + 1e-8)
    ay = -G * M_star * y / (r**3 + 1e-8)

    # BH2 gravity
    dx2 = x - x2
    dy2 = y - y2
    r2p = np.sqrt(dx2**2 + dy2**2) + 1e-8

    ax += -G * M2 * dx2 / r2p**3
    ay += -G * M2 * dy2 / r2p**3

    # PRESSURE 
    ax_p, ay_p = pressure_acceleration(x, y)
    ax += ax_p
    ay += ay_p

    # VISCOSITY (density-scaled)
    rho = local_density_proxy(x, y)
    rho_mean = np.mean(rho)

    nu_local = nu_visc0 * (rho / (rho_mean + 1e-12))

    ax -= nu_local * vx
    ay -= nu_local * vy

    # integrate disk
    vx += ax * dt
    vy += ay * dt
    x += vx * dt
    y += vy * dt

    # accretion
    v_rel = np.sqrt((vx - vx2)**2 + (vy - vy2)**2)
    R_acc = bondi_radius(M2, np.mean(v_rel))

    accreted = r2p < R_acc

    if np.any(accreted):
        dm = np.sum(m[accreted])
        M2 += dm
        m[accreted] *= 1e-6

    return x, y, vx, vy, m


# RUN
x, y, vx, vy, m = initialize_disk(m_particle=1e-4, N=5000)

T = 80000

for i in range(T):
    t = i * dt
    x, y, vx, vy, m = step(x, y, vx, vy, m, t)

    if i % 8000 == 0:
        x2, y2, _, _ = bh2_position(t)

        plt.clf()
        plt.scatter(x, y, s=2, alpha=0.5)
        plt.scatter([0], [0], c='yellow', s=80)
        plt.scatter([x2], [y2], c='red', s=80)
        plt.gca().set_aspect('equal')
        plt.title(f"Disk + pressure + viscous + accretor, step {i}, M2={M2:.3f}")
        plt.xlim(-R_outer, R_outer)
        plt.ylim(-R_outer, R_outer)
        plt.pause(0.01)

plt.show()

print("Final M2:", M2)